In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Setup Device Configuration
# Uses GPU if available for faster training, otherwise defaults to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. Hyperparameters & Data Preparation
# ==========================================
batch_size = 64
learning_rate = 0.001
num_epochs = 5

# Transformations applied to the images: Convert to Tensors and normalize 
# Pixel values are scaled from [0, 255] to a mean of 0.5 and std dev of 0.5
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load the training and test datasets
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# ==========================================
# 3. Defining the CNN Architecture
# ==========================================
class CharacterCNN(nn.Module):
    def __init__(self):
        super(CharacterCNN, self).__init__()
        
        # Convolutional Block 1
        # Input: 1 channel (grayscale), Output: 16 feature maps. Kernel size 3x3.
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        # Reduces dimensions by half (from 28x28 to 14x14)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Convolutional Block 2
        # Input: 16 feature maps, Output: 32 feature maps.
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        # Reduces dimensions by half again (from 14x14 to 7x7)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected (Dense) Layers
        # 32 feature maps of size 7x7 flattened = 32 * 7 * 7 = 1568 inputs
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        # Output layer: 10 outputs corresponding to digits 0-9
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # Pass data through Convolutional Block 1
        x = self.pool1(self.relu1(self.conv1(x)))
        
        # Pass data through Convolutional Block 2
        x = self.pool2(self.relu2(self.conv2(x)))
        
        # Flatten the feature maps into a 1D vector for the fully connected layer
        x = x.view(x.size(0), -1) 
        
        # Pass through fully connected layers
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and move it to the configured device (CPU/GPU)
model = CharacterCNN().to(device)

# ==========================================
# 4. Loss Function and Optimizer
# ==========================================
# CrossEntropyLoss is ideal for multi-class classification
criterion = nn.CrossEntropyLoss()
# Adam optimizer automatically adjusts the learning rate during training
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ==========================================
# 5. Training Loop
# ==========================================
print("Starting Training...")
for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        # Move tensors to the configured device
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass: Compute predicted outputs by passing images to the model
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()       # Compute gradients via backpropagation
        optimizer.step()      # Update network weights
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training Finished!")

# ==========================================
# 6. Evaluation Loop
# ==========================================
model.eval() # Set model to evaluation mode (deactivates dropout/batchnorm if any)
with torch.no_grad(): # Disable gradient calculation for efficiency
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        # Get the index of the highest probability output class
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy of the model on the 10,000 test images: {accuracy:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 5  # You can increase this later for better accuracy

# Transformations (Fixes the EMNIST rotation issue)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.transpose(1, 2).flip(2)), # Flips and rotates to correct alignment
    transforms.Normalize((0.5,), (0.5,))
])

# 1. Download EMNIST Balanced Dataset (47 classes: digits + uppercase/lowercase letters combined)
train_dataset = torchvision.datasets.EMNIST(root='./data', split='balanced', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.EMNIST(root='./data', split='balanced', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Class mapping helper to read the results
class_mapping = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabdefghnqrt"

# 2. Define Architecture (Updated for 47 output classes)
class AlphabetCNN(nn.Module):
    def __init__(self):
        super(AlphabetCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 47) # 47 distinct output classes instead of 10
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

model = AlphabetCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 3. Training Loop
print("Starting EMNIST Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 4. Evaluation Loop
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Finished! Final Accuracy on Letters/Digits: {100 * correct / total:.2f}%")

In [ ]:
import torch
import torch.nn as nn

class HandwritingCRNN(nn.Module):
    def __init__(self, num_classes):
        super(HandwritingCRNN, self).__init__()
        
        # 1. CNN Feature Extractor (Processes the visual handwriting)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2), # Dim: 28x28 -> 14x14
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2), # Dim: 14x14 -> 7x7
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(),
            # Crucial: We only downsample the Height, preserving Width resolution for the sequence
            nn.MaxPool2d(kernel_size=(2, 1)) 
        )
        
        # 2. Linear Bridge: Maps CNN output dimensions to RNN input features
        # Assuming an input height of 28px downsamples to a feature height of ~3px
        self.linear_bridge = nn.Linear(256 * 3, 64) 
        
        # 3. RNN Sequence Processor (Learns language context and letter ordering)
        # Bidirectional LSTM processes text slices from left-to-right and right-to-left
        self.rnn = nn.LSTM(input_size=64, hidden_size=128, num_layers=2, 
                           bidirectional=True, batch_first=True)
        
        # 4. Transcription/Output Layer
        # Output size is multiplied by 2 because Bidirectional LSTM combines forward + backward passes
        self.fc = nn.Linear(128 * 2, num_classes) 

    def forward(self, x):
        # Expected input shape: [Batch Size, 1, Height, Width]
        features = self.cnn(x)
        
        # Reshape spatial features into a time-series sequence for the RNN
        batch, channels, height, width = features.size()
        features = features.view(batch, channels * height, width) 
        features = features.permute(0, 2, 1) # Rearrange to: [Batch, Sequence_Length (Width), Features]
        
        # Pass sequence through the Linear layer and LSTM
        rnn_input = self.linear_bridge(features)
        rnn_output, _ = self.rnn(rnn_input)
        
        # Output raw prediction scores for each character slot
        output = self.fc(rnn_output) 
        return output # Final Shape: [Batch, Sequence_Length, Num_Classes]

# Instantiate model (e.g., 47 classes from EMNIST + 1 extra slot for CTC blank label = 48)
crnn_model = HandwritingCRNN(num_classes=48)
print(crnn_model)

In [ ]:
# 1. Create a dummy "Word" image tensor
# Shape: [Batch Size, Channels, Height, Width]
# We make the width 128 pixels to simulate a wide word image instead of a single square character
mock_word_batch = torch.randn(16, 1, 28, 128).to(device)

print(f"Input Shape (16 words of size 28x128): {mock_word_batch.shape}")

# 2. Make sure the CRNN model is on the same device
crnn_model = crnn_model.to(device)

# 3. Pass the mock word through the pipeline
crnn_model.eval()
with torch.no_grad():
    predictions = crnn_model(mock_word_batch)

# 4. Check the structural output dimensions
print(f"Output Shape from CRNN: {predictions.shape}")
print("-" * 50)
print(f"-> 16: Number of words processed (Batch Size)")
print(f"-> {predictions.shape[1]}: Timesteps/Slices extracted along the horizontal width")
print(f"-> 48: Probability scores for each possible alphabet character (Classes)")

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. Initialize the CTC Loss function
# zero_infinity=True prevents training from crashing if a bad alignment produces infinite loss
criterion_ctc = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(crnn_model.parameters(), lr=0.001)

# 2. Simulated Target Data (What a real dataset loader would pass)
# Let's assume each word in our batch of 16 has a target text length of 5 characters
target_lengths = torch.full(size=(16,), fill_value=5, dtype=torch.long)
# Flat tensor containing the integer class labels for all characters in the batch (16 words * 5 letters = 80 indices)
mock_targets = torch.randint(low=1, high=47, size=(80,), dtype=torch.long)

# The length of the sequence output by the CRNN for each item in the batch (32 slices)
input_lengths = torch.full(size=(16,), fill_value=32, dtype=torch.long)

# =========================================================
# 3. Running a Dummy Training Step
# =========================================================
crnn_model.train()
optimizer.zero_grad()

# Forward pass (Using the mock_word_batch from your previous cell)
# Current predictions shape: [Batch(16), Sequence(32), Classes(48)]
preds = crnn_model(mock_word_batch)

# CRITICAL STEP: PyTorch's CTCLoss expects predictions in the shape:
# [Sequence_Length, Batch_Size, Number_of_Classes]
preds = preds.permute(1, 0, 2) 

# Log-softmax must be applied to the predictions before feeding them to CTCLoss
preds_log_softmax = preds.log_softmax(2)

# Calculate Loss
loss = criterion_ctc(preds_log_softmax, mock_targets, input_lengths, target_lengths)

# Backward pass
loss.backward()
optimizer.step()

print(f"CTC Loss successfully calculated: {loss.item():.4f}")
print("Backward propagation pass complete! Your pipeline is structurally flawless.")

In [ ]:
def ctc_decode(model_output, mapping):
    """
    Greedy Decoder for CTC outputs.
    Collapses consecutive duplicate characters and removes blank tokens.
    """
    # Find the character index with the highest probability for each timestep
    # Shape: [Sequence_Length]
    _, max_indices = torch.max(model_output, dim=-1)
    
    decoded_string = []
    previous_idx = None
    
    for idx in max_indices:
        idx = idx.item()
        # 1. Skip if it is the CTC blank token (0)
        # 2. Skip if it is a consecutive duplicate character
        if idx != 0 and idx != previous_idx:
            # Shift index by -1 because class_mapping index 0 is '0', but CTC blank is 0
            decoded_string.append(mapping[idx - 1])
        previous_idx = idx
        
    return "".join(decoded_string)

# Let's test it on one sequence from your batch
sample_word_prediction = preds[:, 0, :] # Grab the first word's timeline sequence
decoded_word = ctc_decode(sample_word_prediction, class_mapping)

print(f"Decoded Text Result: '{decoded_word}'")

In [ ]:
import gradio as gr
import torch
import torchvision.transforms as transforms
from PIL import Image, ImageOps

# 1. Ensure the trained AlphabetCNN model is in evaluation mode
model.eval()

def preprocess_single_char(pil_image):
    # Convert image to grayscale
    img = pil_image.convert('L')
    
    # Invert colors so the background becomes dark (White letter on Black background)
    img_inverted = ImageOps.invert(img)
    
    # Crop out excess outer whitespace margins to isolate the letter skeleton
    bbox = img_inverted.getbbox()
    if bbox:
        img_inverted = img_inverted.crop(bbox)
    
    # Resize the isolated letter cleanly to 20x20
    img_letter = img_inverted.resize((20, 20))
    
    # Create a solid black 28x28 canvas (matches the exact EMNIST dataset dimensions)
    canvas = Image.new('L', (28, 28), color=0)
    canvas.paste(img_letter, (4, 4)) # Centers the letter with a 4-pixel border padding
    
    # Convert to a standard tensor and normalize (NO manual flips or rotations!)
    transform_pipeline = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    return transform_pipeline(canvas).unsqueeze(0).to(device)

def predict_character(input_image):
    if input_image is None:
        return "Please upload an image."
    try:
        image_tensor = preprocess_single_char(input_image)
        with torch.no_grad():
            outputs = model(image_tensor)
            _, predicted = torch.max(outputs.data, 1)
        
        # Map the predicted index back to our alphabet string
        return class_mapping[predicted.item()]
    except Exception as e:
        return f"Error: {str(e)}"

# Re-launch the character recognizer window
char_interface = gr.Interface(
    fn=predict_character,
    inputs=gr.Image(type="pil", label="Crop a SINGLE Letter or Digit"),
    outputs=gr.Textbox(label="Predicted Character"),
    title="Character Recognition System (Upright Input)"
)
char_interface.launch(share=False)